# AETHER — Mission Control AI · Predictive Health Monitor (S2)
## Global Solution 2026.1 — Aprendizado Profundo com Redes Neurais (Keras)

**Operadora:** Orbital Climate Intelligence (OCI) · Missao **AETHER-1** · Base **Houston Control**
**Tagline:** *AETHER — Do espaco, cuidando da Terra.*

| Integrante | RM |
|------------|----|
| Rogerio Deligi | 561942 |
| Maria Fernanda Garavelli Dantas | 562686 |

**Curso:** 2o ano — Ciencia da Computacao — FIAP
**Subsistema entregue:** **S2 — Predictive Health Monitor** (IA preditiva de falha do AETHER).
**Tipo de rede:** MLP densa (`Dense`) como arquitetura **principal**. O dataset e **tabular**
(telemetria/sensores), portanto `Conv2D` **nao se aplica** — convolucao 2D pressupoe estrutura
espacial de grade (pixels vizinhos de uma imagem), que aqui nao existe. Ainda assim, como a
disciplina e de Redes Convolucionais, **incluimos um experimento de controle com `Conv1D`**
(Secao 7, experimento E7) para comprovar empiricamente que a convolucao **nao** traz ganho sobre
dados sem vizinhanca local — fechando o argumento conceitual com evidencia, nao so com teoria.
A escolha da MLP densa e justificada na secao de EDA e confirmada por esse experimento.

---

### O que o S2 faz na narrativa AETHER
O Predictive Health Monitor le a telemetria de bordo da AETHER-1 (condicoes operacionais +
21 sensores) e classifica o **estado operacional** em 3 niveis, alimentando o **Alert Engine (S3)**:

| target | Estado (brief) | Nivel canonico AETHER (Alert Engine) | Cor |
|:------:|----------------|--------------------------------------|-----|
| 0 | Operacao normal | **NOMINAL** | verde `#22C55E` |
| 1 | Alerta operacional | **ALERTA** | laranja `#FB923C` |
| 2 | Falha critica iminente | **CRITICO** | vermelho `#EF4444` |

> **Base de dados:** usamos **exclusivamente** os arquivos do professor `orbital_train.csv` e
> `orbital_test.csv` (NASA C-MAPSS adaptado — simulacao de degradacao de sistemas aeronauticos).


## 0. Reprodutibilidade e configuracao do ambiente

Fixamos as seeds (`random`, `numpy`, `tensorflow`) para que os resultados sejam reproduziveis e
as comparacoes entre experimentos sejam justas. Centralizamos toda a configuracao no inicio.


In [ ]:
# Configuracao do ambiente — execute esta celula primeiro.
import os, random, time, json, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # reduz ruido de log do TensorFlow

import numpy as np
import pandas as pd

SEED = 42
def set_global_seed(seed=SEED):
    """Fixa todas as fontes de aleatoriedade e limpa a sessao do Keras."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    import tensorflow as tf
    tf.random.set_seed(seed)
    tf.keras.backend.clear_session()

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, Input,
                                      Conv1D, GlobalAveragePooling1D, Reshape)
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Paleta Mission Control (identidade visual canonica AETHER) para os graficos
AETHER = {
    "bg": "#0A0E1A", "panel": "#161C2E", "grid": "#233047",
    "text": "#E5E7EB", "muted": "#9CA3AF",
    "cyan": "#22D3EE", "amber": "#F5A623",
    "NOMINAL": "#22C55E", "ALERTA": "#FB923C", "CRITICO": "#EF4444",
}
CLASS_NAMES = ["NOMINAL (0)", "ALERTA (1)", "CRITICO (2)"]

print("TensorFlow:", tf.__version__)
print("Ambiente configurado. Seed global =", SEED)


## 1. Carregamento da base do professor

Coloque `orbital_train.csv` e `orbital_test.csv` na mesma pasta do notebook
(ou ajuste `DATA_DIR`). No Google Colab, o trecho comentado faz o upload manual.

A base ja vem **separada em treino e teste pelo professor** — vamos respeitar essa separacao e
nunca usar o teste para ajustar nada (nem o scaler, nem a escolha de hiperparametros).


In [ ]:
# --- Caminho dos dados -------------------------------------------------
# Local (Jupyter): os CSVs ficam na subpasta dados/ ao lado do notebook.
# Colab: descomente o bloco de upload abaixo.
from pathlib import Path
DATA_DIR = Path("dados") if Path("dados/orbital_train.csv").exists() else Path(".")

# --- Upload no Google Colab (descomente se estiver no Colab) -----------
# from google.colab import files
# print("Selecione orbital_train.csv e orbital_test.csv:")
# up = files.upload()
# DATA_DIR = Path(".")

train_df = pd.read_csv(DATA_DIR / "orbital_train.csv")
test_df  = pd.read_csv(DATA_DIR / "orbital_test.csv")

print("Treino:", train_df.shape, "| Teste:", test_df.shape)
print("\\nColunas:", list(train_df.columns))
train_df.head()


## 2. Analise exploratoria (EDA) breve

Objetivo: entender a estrutura dos dados antes de modelar. Verificamos:
1. **Tipos e valores ausentes** — a rede nao aceita `NaN`.
2. **Distribuicao das classes** — ha desbalanceamento? (impacta a leitura da accuracy).
3. **Colunas constantes** — sensores sem variancia nao informam nada e podem ser removidos.
4. **A coluna `RUL`** — *Remaining Useful Life* (vida util remanescente). Inspecionamos a relacao
   `RUL` vs `target` porque ela e a chave de uma decisao de modelagem critica (vazamento de alvo).


In [ ]:
# 2.1 Tipos, ausentes e estatisticas
print(">>> Valores ausentes por coluna (esperado: 0):")
print(train_df.isna().sum().sum(), "ausentes no treino |",
      test_df.isna().sum().sum(), "ausentes no teste")

print("\\n>>> Estatisticas descritivas (treino):")
display(train_df.describe().T[["mean", "std", "min", "max"]])


In [ ]:
# 2.2 Distribuicao das classes (target) — treino vs teste
dist = pd.DataFrame({
    "treino": train_df["target"].value_counts().sort_index(),
    "teste":  test_df["target"].value_counts().sort_index(),
})
dist["treino_%"] = (dist["treino"] / dist["treino"].sum() * 100).round(1)
dist["teste_%"]  = (dist["teste"]  / dist["teste"].sum()  * 100).round(1)
dist.index = CLASS_NAMES
print(">>> Distribuicao das classes:")
display(dist)

# Grafico de barras com a paleta Mission Control
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor(AETHER["bg"]); ax.set_facecolor(AETHER["panel"])
bars_colors = [AETHER["NOMINAL"], AETHER["ALERTA"], AETHER["CRITICO"]]
ax.bar(CLASS_NAMES, dist["treino"], color=bars_colors, edgecolor=AETHER["grid"])
ax.set_title("AETHER S2 — Distribuicao do estado operacional (treino)", color=AETHER["text"])
ax.tick_params(colors=AETHER["muted"]); ax.set_ylabel("Amostras", color=AETHER["text"])
for s in ax.spines.values(): s.set_color(AETHER["grid"])
plt.tight_layout(); plt.show()


In [ ]:
# 2.3 Colunas constantes (variancia zero) — candidatas a remocao
nunique = train_df.nunique()
constantes = nunique[nunique <= 1].index.tolist()
print(">>> Colunas constantes no treino (sem informacao):")
print(constantes)


### 2.4 Decisao critica: `RUL` como feature → **vazamento de alvo (target leakage)**

`RUL` = *Remaining Useful Life* (ciclos restantes ate a falha). Inspecionando a relacao entre
`RUL` e `target`, observamos que o **alvo e uma binarizacao deterministica do RUL**: as faixas de
`RUL` mapeiam exatamente para cada classe e **nao se sobrepoem**. Valores reais inspecionados nos
dois arquivos do professor (a celula 2.5 reproduz a evidencia):

| Classe | Nivel AETHER | Faixa de `RUL` (treino) | Faixa de `RUL` (teste) |
|:------:|--------------|:-----------------------:|:----------------------:|
| 0 | NOMINAL  | 121 – 361 | 121 – 356 |
| 1 | ALERTA   | 51 – 120  | 51 – 120  |
| 2 | CRITICO  | 0 – 50    | 0 – 50    |

Os cortes (`RUL > 120` → 0, `51 ≤ RUL ≤ 120` → 1, `RUL ≤ 50` → 2) sao **identicos** em treino e
teste. Logo, **usar `RUL` como entrada vaza a resposta** — o modelo so precisa "ler o gabarito" e
acerta de forma artificial (~100%).

Isso e um caso classico de **data leakage**: a metrica fica inflada e o modelo nao aprende o
padrao real dos **sensores**, que e o que importa para o Predictive Health Monitor em operacao
(em voo, o RUL exato e desconhecido — e justamente o que queremos inferir).

> **Como tratamos isto (cumprindo o pedido do brief):** rodamos **os dois cenarios** e comparamos.
> - **Cenario A (COM RUL):** demonstra o vazamento — accuracy ~perfeita, sem valor preditivo real.
> - **Cenario B (SEM RUL):** modelagem honesta — o modelo aprende dos sensores. **E o cenario
>   oficial** de todos os experimentos do S2.


In [ ]:
# 2.5 Evidencia do vazamento: faixa de RUL por classe (treino E teste)
for nome_df, df in [("TREINO", train_df), ("TESTE", test_df)]:
    rul_por_classe = df.groupby("target")["RUL"].agg(["min", "max", "mean", "count"])
    rul_por_classe.index = CLASS_NAMES
    print(f">>> Faixa de RUL por classe ({nome_df}) — note as faixas disjuntas:")
    display(rul_por_classe.round(2))

# Verificacao programatica: as faixas se sobrepoem entre classes vizinhas?
def faixas_disjuntas(df):
    g = df.groupby("target")["RUL"].agg(["min", "max"])
    # classe 2 (RUL baixo) vs classe 1, e classe 1 vs classe 0 (RUL alto)
    sep_2_1 = g.loc[2, "max"] < g.loc[1, "min"]
    sep_1_0 = g.loc[1, "max"] < g.loc[0, "min"]
    return bool(sep_2_1 and sep_1_0)

print("\\nFaixas de RUL totalmente disjuntas? "
      f"treino={faixas_disjuntas(train_df)} | teste={faixas_disjuntas(test_df)}")
print("Interpretacao: RUL alto => NOMINAL | RUL medio => ALERTA | RUL baixo => CRITICO.")
print("Faixas disjuntas => target e funcao deterministica de RUL => VAZAMENTO se usado como feature.")


## 3. Preparacao dos dados (pipeline)

Passos:
1. **Separar X e y** (entradas vs alvo).
2. **Remover colunas constantes** (sem informacao) e definir as duas listas de features:
   **COM RUL** (cenario A) e **SEM RUL** (cenario B — oficial).
3. **Split de validacao**: separamos 20% do **treino** para validacao (estratificado). O **teste do
   professor permanece intocado** ate a avaliacao final.
4. **StandardScaler**: padronizacao (media 0, desvio 1). **O `fit` e feito SO no treino** e
   aplicamos (`transform`) em validacao e teste — evita vazamento de informacao do teste.
5. **One-hot** do alvo com `to_categorical` (3 classes) para usar `categorical_crossentropy`.

> **Fatos verificados na base (a celula 3.1 reproduz):** 7 colunas constantes
> (`op_setting_3`, `sensor_1`, `sensor_5`, `sensor_10`, `sensor_16`, `sensor_18`, `sensor_19`)
> sao removidas. Restam **17 features** no cenario oficial SEM RUL (e **18** no cenario COM RUL,
> so para a demonstracao de vazamento). Base sem valores ausentes (0 `NaN`).


In [ ]:
# 3.1 Definicao de features (X) e alvo (y)
TARGET = "target"

# Colunas constantes detectadas na EDA sao removidas (nao informam nada).
const_cols = [c for c in train_df.columns if train_df[c].nunique() <= 1 and c != TARGET]

# Cenario B (OFICIAL): SEM RUL e sem colunas constantes
features_sem_rul = [c for c in train_df.columns
                    if c not in (["RUL", TARGET] + const_cols)]
# Cenario A (so para demonstrar o vazamento): COM RUL
features_com_rul = features_sem_rul + ["RUL"]

print("Colunas constantes removidas:", const_cols)
print("\\nN features SEM RUL (oficial):", len(features_sem_rul))
print("N features COM RUL (demo vazamento):", len(features_com_rul))

y_train_full = train_df[TARGET].values
y_test       = test_df[TARGET].values
NUM_CLASSES  = 3


In [ ]:
# 3.2 Funcao de preparacao: split de validacao + StandardScaler (fit so no treino) + one-hot
def preparar(features):
    """Retorna X/y de treino, validacao e teste ja escalados e com y one-hot.
    O scaler e ajustado SOMENTE no treino (evita vazamento)."""
    X_full = train_df[features].values.astype("float32")
    X_test = test_df[features].values.astype("float32")

    # Split estratificado de validacao (20% do treino)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full)

    # StandardScaler: fit SO no treino, transform nos demais
    scaler = StandardScaler().fit(X_tr)
    X_tr  = scaler.transform(X_tr)
    X_val = scaler.transform(X_val)
    X_te  = scaler.transform(X_test)

    # One-hot encoding do alvo
    Y_tr  = to_categorical(y_tr,  NUM_CLASSES)
    Y_val = to_categorical(y_val, NUM_CLASSES)
    Y_te  = to_categorical(y_test, NUM_CLASSES)
    return (X_tr, Y_tr, X_val, Y_val, X_te, Y_te, y_test)

# Conjunto OFICIAL (sem RUL)
X_tr, Y_tr, X_val, Y_val, X_te, Y_te, y_te = preparar(features_sem_rul)
INPUT_DIM = X_tr.shape[1]
print("Shapes (cenario oficial SEM RUL):")
print("  X_tr:", X_tr.shape, "| X_val:", X_val.shape, "| X_te:", X_te.shape)
print("  Y_tr:", Y_tr.shape, "(one-hot, 3 classes)")
print("  INPUT_DIM =", INPUT_DIM)


## 4. Utilitarios de construcao e treino

Encapsulamos o ciclo *montar → compilar → treinar → avaliar* numa MLP densa parametrizada.
Assim cada experimento **isola uma unica variavel** (otimizador, profundidade, ativacao, batch,
regularizacao...) mantendo o resto constante — metodologia de comparacao controlada.

**Rede base (baseline) do S2:** camadas densas `[64, 32]`, ativacao ReLU, otimizador Adam
(`lr=1e-3`), `batch_size=64`, ate 60 epocas com `EarlyStopping` (paciencia 8, restaura o melhor
peso), saida `softmax` de 3 neuronios, perda `categorical_crossentropy`.

Alem do construtor de MLP densa, definimos tambem um `build_cnn1d` — uma **CNN 1D de controle**
usada apenas no experimento E7 (Secao 7) para comparar, com numeros, a MLP densa contra uma
arquitetura convolucional sobre os mesmos dados tabulares.


In [ ]:
# 4.1 Construtor de MLP densa parametrizada (NAO ha Conv2D — dado tabular)
RESULTS = {}   # guarda history + metricas de TODOS os experimentos

def build_mlp(hidden=(64, 32), activation="relu", dropout=0.0, l2_lambda=0.0,
              batch_norm=False, input_dim=None, num_classes=NUM_CLASSES):
    """Monta uma MLP densa. batch_norm aplica BatchNormalization apos cada Dense."""
    input_dim = input_dim or INPUT_DIM
    reg = l2(l2_lambda) if l2_lambda > 0 else None
    model = Sequential(name="AETHER_S2_PredictiveHealthMonitor")
    model.add(Input(shape=(input_dim,)))
    for u in hidden:
        model.add(Dense(u, activation=None, kernel_regularizer=reg))
        if batch_norm:
            model.add(BatchNormalization())
        model.add(keras.layers.Activation(activation))
        if dropout > 0:
            model.add(Dropout(dropout))
    model.add(Dense(num_classes, activation="softmax"))
    return model


def build_cnn1d(filters=(32, 16), kernel_size=2, input_dim=None, num_classes=NUM_CLASSES):
    """Monta uma CNN 1D de CONTROLE. Trata o vetor de features como uma 'sequencia'
    (input_dim passos x 1 canal) so para TESTAR se a convolucao ajuda. Como os dados sao
    tabulares (sem vizinhanca local entre colunas), esperamos que NAO supere a MLP densa —
    esse experimento existe para comprovar essa hipotese com numeros, nao com teoria."""
    input_dim = input_dim or INPUT_DIM
    model = Sequential(name="AETHER_S2_CNN1D_controle")
    model.add(Input(shape=(input_dim,)))
    model.add(Reshape((input_dim, 1)))                      # (features, 1 canal)
    for f in filters:
        model.add(Conv1D(f, kernel_size=kernel_size, activation="relu", padding="same"))
    model.add(GlobalAveragePooling1D())
    model.add(Dense(32, activation="relu"))
    model.add(Dense(num_classes, activation="softmax"))
    return model


def make_optimizer(name="adam", lr=1e-3):
    name = name.lower()
    if name == "adam":    return Adam(learning_rate=lr)
    if name == "sgd":     return SGD(learning_rate=lr, momentum=0.9)
    if name == "rmsprop": return RMSprop(learning_rate=lr)
    raise ValueError(f"Otimizador desconhecido: {name}")


def train_and_eval(name, model, optimizer="adam", lr=1e-3, epochs=60,
                   batch_size=64, patience=8, verbose=0,
                   data=(X_tr, Y_tr, X_val, Y_val, X_te, Y_te, y_te)):
    """Compila, treina com EarlyStopping e avalia em treino/val/teste.
    Salva tudo em RESULTS[name] e devolve o dicionario de metricas."""
    Xtr, Ytr, Xval, Yval, Xte, Yte, yte = data
    set_global_seed(SEED)
    opt = make_optimizer(optimizer, lr) if isinstance(optimizer, str) else optimizer
    model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
    es = EarlyStopping(monitor="val_loss", patience=patience,
                       restore_best_weights=True, verbose=0)
    t0 = time.time()
    hist = model.fit(Xtr, Ytr, validation_data=(Xval, Yval),
                     epochs=epochs, batch_size=batch_size,
                     callbacks=[es], verbose=verbose)
    dt = time.time() - t0

    tr_loss,  tr_acc  = model.evaluate(Xtr,  Ytr,  verbose=0)
    val_loss, val_acc = model.evaluate(Xval, Yval, verbose=0)
    te_loss,  te_acc  = model.evaluate(Xte,  Yte,  verbose=0)
    gap = tr_acc - val_acc   # indicador de overfitting

    RESULTS[name] = {
        "history": hist.history, "model": model,
        "epocas_rodadas": len(hist.history["loss"]),
        "train_acc": tr_acc, "val_acc": val_acc, "test_acc": te_acc,
        "train_loss": tr_loss, "val_loss": val_loss, "test_loss": te_loss,
        "gap_overfit": gap, "tempo_s": dt,
    }
    print(f"[{name:28s}] val_acc={val_acc:.4f} test_acc={te_acc:.4f} "
          f"gap={gap:+.4f} epocas={len(hist.history['loss'])} ({dt:.1f}s)")
    return RESULTS[name]


In [ ]:
# 4.2 Funcoes de visualizacao (curvas de treino e matriz de confusao) com a paleta AETHER
import matplotlib.pyplot as plt
import seaborn as sns

def plot_curvas(name):
    """Plota loss e accuracy (treino vs validacao) de um experimento."""
    h = RESULTS[name]["history"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor(AETHER["bg"])
    for ax, (m, titulo) in zip(axes, [("loss", "Loss"), ("accuracy", "Accuracy")]):
        ax.set_facecolor(AETHER["panel"])
        ax.plot(h[m], color=AETHER["cyan"], label="treino")
        ax.plot(h["val_" + m], color=AETHER["amber"], label="validacao")
        ax.set_title(f"{name} — {titulo}", color=AETHER["text"])
        ax.set_xlabel("epoca", color=AETHER["muted"])
        ax.tick_params(colors=AETHER["muted"])
        ax.grid(color=AETHER["grid"], alpha=0.3)
        for s in ax.spines.values(): s.set_color(AETHER["grid"])
        ax.legend(facecolor=AETHER["panel"], labelcolor=AETHER["text"])
    plt.tight_layout(); plt.show()


def plot_matriz_confusao(name):
    """Matriz de confusao do experimento no conjunto de TESTE do professor."""
    model = RESULTS[name]["model"]
    y_pred = model.predict(X_te, verbose=0).argmax(axis=1)
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5)); fig.patch.set_facecolor(AETHER["bg"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="mako",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
                cbar_kws={"label": "amostras"})
    ax.set_title(f"AETHER S2 — Matriz de confusao ({name})", color=AETHER["text"])
    ax.set_xlabel("Previsto", color=AETHER["text"]); ax.set_ylabel("Real", color=AETHER["text"])
    plt.tight_layout(); plt.show()
    print("\\n>>> classification_report (TESTE do professor):")
    print(classification_report(y_te, y_pred, target_names=CLASS_NAMES, digits=4))
    return cm


## 5. Modelo base (baseline) — referencia dos experimentos

Treinamos a rede base no cenario **oficial (SEM RUL)**. Todos os experimentos da Secao 7
sao comparados contra este baseline.


In [ ]:
# 5.1 Baseline: [64, 32] ReLU, Adam lr=1e-3, batch 64
set_global_seed(SEED)
base = build_mlp(hidden=(64, 32), activation="relu", dropout=0.0,
                 l2_lambda=0.0, batch_norm=False)
base.summary()


In [ ]:
# 5.2 Treina e avalia o baseline
train_and_eval("baseline (64-32, Adam)", base, optimizer="adam",
               lr=1e-3, epochs=60, batch_size=64, patience=8, verbose=0)
plot_curvas("baseline (64-32, Adam)")


## 6. Demonstracao controlada do vazamento de alvo (COM RUL vs SEM RUL)

Aqui rodamos o **mesmo baseline** com a feature `RUL` incluida (cenario A). Esperamos
accuracy artificialmente proxima de 100% — evidencia do vazamento. **Este modelo NAO e usado
no S2 final**; serve apenas para provar, com numeros, por que excluimos o RUL.


In [ ]:
# 6.1 Prepara o conjunto COM RUL e treina o mesmo baseline
data_com_rul = preparar(features_com_rul)
INPUT_DIM_RUL = data_com_rul[0].shape[1]

set_global_seed(SEED)
base_rul = build_mlp(hidden=(64, 32), activation="relu",
                     input_dim=INPUT_DIM_RUL)
train_and_eval("DEMO vazamento (COM RUL)", base_rul, optimizer="adam",
               lr=1e-3, epochs=60, batch_size=64, patience=8, verbose=0,
               data=data_com_rul)

print("\\nComparacao direta:")
print(f"  SEM RUL (oficial) -> test_acc = {RESULTS['baseline (64-32, Adam)']['test_acc']:.4f}")
print(f"  COM RUL (vazamento) -> test_acc = {RESULTS['DEMO vazamento (COM RUL)']['test_acc']:.4f}")
print("Se COM RUL ~ 1.00, confirma-se o vazamento: o modelo le o gabarito, nao aprende dos sensores.")


## 7. Bateria de experimentos comparativos (cenario oficial SEM RUL)

Cada experimento varia **uma dimensao** e mantem o resto igual ao baseline, para isolar o efeito.

| # | Experimento | Variavel manipulada |
|---|-------------|---------------------|
| E1 | Otimizadores | Adam vs SGD(momentum) vs RMSprop |
| E2 | Profundidade x largura | [32] · [64,32] · [128,64,32] · [256,128,64,32] |
| E3 | Funcao de ativacao | ReLU vs tanh vs sigmoid |
| E4 | Batch size | 16 vs 64 vs 256 |
| E5 | Regularizacao | baseline vs Dropout vs L2 vs Dropout+BatchNorm |
| E6 | Epocas x EarlyStopping | sem ES (fixo) vs com ES (paciencia 8) |
| E7 | **Arquitetura: MLP densa vs CNN 1D** | MLP `[64,32]` vs `Conv1D` (controle convolucional) |

Todas as historias ficam em `RESULTS`, de onde extraimos a tabela comparativa consolidada.

> **Sobre o E7 (relevante para a disciplina DL-CNN):** embora o dado seja tabular e a MLP densa
> seja a arquitetura correta, incluimos uma **CNN 1D de controle** que trata o vetor de features
> como uma sequencia. O objetivo nao e ganhar accuracy, e sim **comprovar empiricamente** que a
> convolucao nao agrega quando nao ha vizinhanca local entre as colunas — transformando a
> justificativa teorica "por que nao CNN" em uma conclusao baseada em evidencia.


In [ ]:
# E1 — Otimizadores (Adam / SGD+momentum / RMSprop)
print(">>> E1: Otimizadores")
for opt in ["adam", "sgd", "rmsprop"]:
    set_global_seed(SEED)
    m = build_mlp(hidden=(64, 32), activation="relu")
    train_and_eval(f"E1 {opt}", m, optimizer=opt, lr=1e-3,
                   epochs=60, batch_size=64, patience=8)


In [ ]:
# E2 — Profundidade x largura
print(">>> E2: Arquitetura (profundidade x largura)")
arqs = {
    "E2 [32]":            (32,),
    "E2 [64,32]":         (64, 32),
    "E2 [128,64,32]":     (128, 64, 32),
    "E2 [256,128,64,32]": (256, 128, 64, 32),
}
for nome, hid in arqs.items():
    set_global_seed(SEED)
    m = build_mlp(hidden=hid, activation="relu")
    train_and_eval(nome, m, optimizer="adam", lr=1e-3,
                   epochs=60, batch_size=64, patience=8)


In [ ]:
# E3 — Funcao de ativacao
print(">>> E3: Ativacao")
for act in ["relu", "tanh", "sigmoid"]:
    set_global_seed(SEED)
    m = build_mlp(hidden=(64, 32), activation=act)
    train_and_eval(f"E3 {act}", m, optimizer="adam", lr=1e-3,
                   epochs=60, batch_size=64, patience=8)


In [ ]:
# E4 — Batch size
print(">>> E4: Batch size")
for bs in [16, 64, 256]:
    set_global_seed(SEED)
    m = build_mlp(hidden=(64, 32), activation="relu")
    train_and_eval(f"E4 batch={bs}", m, optimizer="adam", lr=1e-3,
                   epochs=60, batch_size=bs, patience=8)


In [ ]:
# E5 — Regularizacao (Dropout / L2 / Dropout+BatchNorm)
print(">>> E5: Regularizacao")
set_global_seed(SEED); m = build_mlp(hidden=(128, 64, 32))
train_and_eval("E5 sem regularizacao", m, optimizer="adam", patience=8)

set_global_seed(SEED); m = build_mlp(hidden=(128, 64, 32), dropout=0.3)
train_and_eval("E5 Dropout 0.3", m, optimizer="adam", patience=8)

set_global_seed(SEED); m = build_mlp(hidden=(128, 64, 32), l2_lambda=1e-3)
train_and_eval("E5 L2 1e-3", m, optimizer="adam", patience=8)

set_global_seed(SEED); m = build_mlp(hidden=(128, 64, 32), dropout=0.3, batch_norm=True)
train_and_eval("E5 Dropout+BatchNorm", m, optimizer="adam", patience=8)


In [ ]:
# E6 — Efeito do EarlyStopping (sem ES, epocas fixas vs com ES)
print(">>> E6: EarlyStopping vs epocas fixas")
# Sem ES: paciencia gigante => roda todas as epocas
set_global_seed(SEED); m = build_mlp(hidden=(128, 64, 32))
train_and_eval("E6 sem ES (60 ep fixas)", m, optimizer="adam",
               epochs=60, batch_size=64, patience=10**6)

set_global_seed(SEED); m = build_mlp(hidden=(128, 64, 32))
train_and_eval("E6 com ES (pac 8)", m, optimizer="adam",
               epochs=60, batch_size=64, patience=8)


In [ ]:
# E7 — MLP densa vs CNN 1D de controle (relevante p/ a disciplina DL-CNN)
print(">>> E7: MLP densa vs CNN 1D (controle convolucional)")
# Referencia: a propria MLP baseline ja treinada (mesmos dados SEM RUL).
set_global_seed(SEED)
cnn = build_cnn1d(filters=(32, 16), kernel_size=2)
cnn.summary()
train_and_eval("E7 CNN1D (controle)", cnn, optimizer="adam",
               lr=1e-3, epochs=60, batch_size=64, patience=8)

mlp_ref = RESULTS["baseline (64-32, Adam)"]["val_acc"]
cnn_val = RESULTS["E7 CNN1D (controle)"]["val_acc"]
print("\\nComparacao MLP densa vs CNN 1D (val_acc):")
print(f"  MLP densa [64,32] (baseline) -> val_acc = {mlp_ref:.4f}")
print(f"  CNN 1D (controle)            -> val_acc = {cnn_val:.4f}")
print("Leitura esperada: a CNN 1D NAO deve superar a MLP densa de forma relevante, pois nao ha")
print("vizinhanca local entre colunas tabulares => confirma que a convolucao nao se aplica aqui.")


## 8. Tabela comparativa consolidada

Reunimos todos os experimentos num unico `DataFrame`, ordenado pela accuracy de validacao.
A coluna `gap_overfit` (train_acc - val_acc) sinaliza overfitting (quanto maior, mais o modelo
decorou o treino).


In [ ]:
# 8.1 Tabela comparativa de todos os experimentos
linhas = []
for nome, r in RESULTS.items():
    linhas.append({
        "experimento": nome,
        "val_acc": round(r["val_acc"], 4),
        "test_acc": round(r["test_acc"], 4),
        "val_loss": round(r["val_loss"], 4),
        "gap_overfit": round(r["gap_overfit"], 4),
        "epocas": r["epocas_rodadas"],
        "tempo_s": round(r["tempo_s"], 1),
    })
tabela = pd.DataFrame(linhas).sort_values("val_acc", ascending=False).reset_index(drop=True)
print(">>> Tabela comparativa (ordenada por val_acc):")
display(tabela)

# Salva a tabela para citar no relatorio
tabela.to_csv("tabela_comparativa_experimentos.csv", index=False)
print("\\nTabela salva em tabela_comparativa_experimentos.csv")


## 9. Modelo final do S2 — avaliacao completa

Selecionamos a melhor configuracao observada (excluindo a demo COM RUL) e a avaliamos a fundo:
curvas de treino, matriz de confusao e `classification_report` no **teste do professor**.

> Ajuste `MELHOR` abaixo conforme o vencedor da sua execucao (a celula sugere automaticamente
> o melhor `val_acc` que **nao** seja a demo de vazamento).


In [ ]:
# 9.1 Seleciona o melhor (ignorando a demo COM RUL)
candidatos = {k: v for k, v in RESULTS.items() if "COM RUL" not in k}
MELHOR = max(candidatos, key=lambda k: candidatos[k]["val_acc"])
print("Melhor configuracao (por val_acc, sem vazamento):", MELHOR)
print(f"  val_acc={RESULTS[MELHOR]['val_acc']:.4f} | test_acc={RESULTS[MELHOR]['test_acc']:.4f}")


In [ ]:
# 9.2 Curvas, matriz de confusao e classification_report do modelo final
plot_curvas(MELHOR)
cm = plot_matriz_confusao(MELHOR)


## 10. Analise critica (preencher/confirmar com os numeros da SUA execucao)

> As celulas acima produzem os numeros reais. Abaixo esta o roteiro de analise critica que a
> rubrica exige (peso 4,0). Confirme cada `[NUMERO]` com a saida da sua execucao.

### 10.1 Comportamento da loss
- A `loss` de treino cai de forma [monotonica/ruidosa]; a `val_loss` [acompanha / descola] da de
  treino. O ponto onde `val_loss` para de cair (e o `EarlyStopping` dispara) indica o melhor
  trade-off vies-variancia. No baseline isso ocorreu por volta da epoca **[NUMERO]**.

### 10.2 Comportamento da accuracy
- A accuracy de validacao estabiliza em torno de **[NUMERO]%** no cenario oficial (SEM RUL).
  Como as 3 classes nao sao perfeitamente separaveis so pelos sensores (degradacao e gradual),
  esse patamar e coerente — e bem diferente do ~100% artificial do cenario COM RUL.

### 10.3 Overfitting
- O `gap_overfit` (train_acc - val_acc) foi **[NUMERO]** no baseline. Em E5, o **Dropout/L2**
  [reduziu / nao alterou] esse gap, ao custo de [pequena queda / nenhuma mudanca] na val_acc —
  o classico trade-off de regularizacao. A arquitetura mais profunda (E2 [256,128,64,32])
  [aumentou / nao aumentou] o gap, sinal de [maior capacidade de decorar / capacidade ociosa].

### 10.4 Estabilidade do treinamento
- **Otimizadores (E1):** Adam e RMSprop convergiram [mais rapido / de forma mais estavel] que o
  SGD; o SGD precisou de [mais epocas / lr maior] e apresentou curva [mais ruidosa].
- **Batch size (E4):** batch 16 deu passos mais [ruidosos mas regularizantes]; batch 256
  [acelerou por epoca mas convergiu mais devagar / suavizou a curva]. Valores: val_acc
  16=[NUMERO], 64=[NUMERO], 256=[NUMERO].
- **BatchNorm (E5):** [estabilizou / acelerou] a convergencia.

### 10.5 Impacto das escolhas (sintese)
- **Maior impacto positivo:** [otimizador Adam / profundidade adequada / EarlyStopping] —
  justificar com a tabela da Secao 8.
- **Regularizacao:** necessaria? [Sim/Nao] — depende de quanto gap o baseline mostrou.
- **RUL:** **excluido** do modelo oficial por vazamento (Secao 6). Mantemos a modelagem honesta:
  o S2 preve a partir dos **sensores**, como em operacao real.

### 10.6 Diferencas entre os modelos e conclusao
- O ranking final (tabela Secao 8) mostra que **[MODELO VENCEDOR]** entregou a melhor val_acc
  **[NUMERO]%** com test_acc **[NUMERO]%** e gap controlado. Conclusao da equipe: para o
  Predictive Health Monitor (S2) do AETHER, uma MLP densa [rasa/media] com Adam, EarlyStopping
  e [Dropout/L2 se necessario] e suficiente e robusta; aumentar profundidade alem disso so
  adiciona custo sem ganho. O foco — pedido pelo brief — foi **compreender o comportamento da
  rede**, nao maximizar accuracy.

### 10.7 MLP densa vs CNN 1D (E7) — fechando o argumento convolucional
- A disciplina e de Redes Convolucionais, entao **testamos** uma `Conv1D` de controle sobre os
  mesmos dados (E7). A val_acc da CNN 1D foi **[NUMERO]%**, contra **[NUMERO]%** da MLP densa
  baseline. Como esperado, a convolucao **[nao superou / superou marginalmente]** a MLP: os dados
  sao **tabulares**, sem vizinhanca local entre colunas, entao o nucleo convolucional nao encontra
  um padrao espacial coerente para explorar. Isso **confirma com evidencia** a escolha teorica da
  MLP densa — exatamente o tipo de "dominio do conceito" que o brief valoriza, e nao apenas uma
  afirmacao de que "CNN nao se aplica".

---

### Identificacao
**RM561942 — Rogerio Deligi · RM562686 — Maria Fernanda Garavelli Dantas**
AETHER — Mission Control AI · S2 Predictive Health Monitor · Global Solution 2026.1
